# Overview

This notebook contains the data cleaning code for the **Net Overseas Migration**, **Value of Residential Building Work Done** and **Value of Residential Building Work Commenced** datasets.

Information about the **Net Overseas Migration** dataset:
1. This specific Excel file has multiple files within it (tabs), denoting the different years in the Reference period.
2. These "wafers" (as the ABS calls them) will need to be handled separately.
3. Luckily, each wafer follows an identical format so the same cleaning code can be used for each.
4. The plan will be to eventually combine the data from all wafers into a single DataFrame that contains all the information. This can then be saved to disk and used in another notebook for EDA and visualisation.

Since each wafer follows the same format, the cleaning steps will be the same. So we will figure out the cleaning steps for one of the wafers, and iteratively apply this to all wafers.

In [1]:
import pandas as pd
data = pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0)

data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Net Overseas Migration (1),NaN,NaN,NaN,NaN
1,Reference period by State of residence by Dire...,NaN,NaN,NaN,NaN
2,Counting: Persons,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,Filters:,NaN,NaN,NaN,NaN
5,Default Summation,Persons ((x1)),NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN
7,2006,NaN,NaN,NaN,NaN
8,Direction of migration (2),NaN,Arrival,Departure,Total
9,NaN,State of residence,NaN,NaN,NaN


### Preprocessing steps to take
1. [x] Get rid of the header and footer rows (ABS metadata, not relevant to our analysis)
2. [x] Change the column and indices to what is correct (we need the columns to be the direction of migration and indices to be the states and territories)
3. [x] Get rid of any columns that are not needed. (metadata or Excel workbook formatting)

In [2]:
# we will define a cleaning function which takes in a raw dataframe and returns a cleaned version.
def clean_data(raw_data, print_output=False):
    raw_data_cleaned = raw_data.copy()
    data_year = raw_data_cleaned.iloc[7, 0]
    raw_data_cleaned.columns = raw_data_cleaned.loc[8]
    raw_data_cleaned = raw_data_cleaned.rename_axis('Direction of migration', axis='columns')
    raw_data_cleaned.index = raw_data_cleaned.iloc[:, 1]
    raw_data_cleaned = raw_data_cleaned.iloc[10:25, 2:]
    raw_data_cleaned['Arrival'] = raw_data_cleaned['Arrival'].astype(int)
    raw_data_cleaned['Departure'] = raw_data_cleaned['Departure'].astype(int)
    raw_data_cleaned['Total'] = raw_data_cleaned['Total'].astype(int)
    # this column will be important for when we merge DataFrames
    raw_data_cleaned['Year'] = data_year
    if print_output:
        print('=== DATASET INFORMATION ===')
        print('Dataset Type:', type(raw_data_cleaned))
        print('Data Shape:', raw_data_cleaned.shape)
        print()
        print('Data types present in dataset:')
        print(raw_data_cleaned.dtypes)
        print()
        print(raw_data_cleaned.describe())
    return raw_data_cleaned

In [3]:
# Sample output of the data from one of the sheets.
sample = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=0))
sample

Direction of migration,Arrival,Departure,Total,Year
nan,,,,
New South Wales,75020,-39100,35920,2006
Victoria,51670,-23810,27860,2006
Queensland,41240,-21250,19980,2006
South Australia,11820,-4640,7180,2006
Western Australia,26430,-11520,14910,2006
Tasmania,1750,-950,800,2006
Northern Territory,2330,-2080,260,2006
Australian Capital Territory,3170,-2500,670,2006
Not Stated,0,0,0,2006


In [4]:
all_frames = []
for i in range(0, 20):
    frame_i = clean_data(pd.read_excel('../data/net_overseas_migration_location_specific.xlsx', sheet_name=i))
    all_frames.append(frame_i)

cleaned_data = pd.concat(all_frames)
cleaned_data['State-Territory'] = cleaned_data.index
cleaned_data.reset_index(drop=True, inplace=True)
cleaned_data

Direction of migration,Arrival,Departure,Total,Year,State-Territory
0,75020,-39100,35920,2006,New South Wales
1,51670,-23810,27860,2006,Victoria
2,41240,-21250,19980,2006,Queensland
3,11820,-4640,7180,2006,South Australia
4,26430,-11520,14910,2006,Western Australia
...,...,...,...,...,...
295,10,-10,10,2025,Christmas Island
296,0,0,0,2025,Jervis Bay
297,0,0,0,2025,Cocos (Keeling) Islands
298,0,0,0,2025,Other Territories


In [5]:
cleaned_data.groupby('Year')['Total'].sum()

Year
2006     215150
2007     488070
2008     631400
2009     493810
2010     344070
2011     412480
2012     480500
2013     416740
2014     364680
2015     373450
2016     487670
2017     483320
2018     504470
2019     495240
2020      -9930
2021      18610
2022     875730
2023    1061230
2024     659900
2025     320320
Name: Total, dtype: int64

In [6]:
cleaned_data.to_csv('../data/migration_data.csv')

# Summary
1. [x] Extracted each sheet from the Excel file.
2. [x] Got rid of all headers and footers, as well as unnecessary columns.
3. [x] Concatenated all the cleaned DataFrames together.
4. [x] Migration data is now ready to be analysed.

# Next: Cleaning the median price of houses dataset

In [13]:
house_data = pd.read_excel('../data/median_price_and_number_of_transfers.xlsx', sheet_name=1)
house_data

,Unnamed: 0,Median Price of Established House Transfers (Unstratified) ; Sydney ;,Median Price of Established House Transfers (Unstratified) ; Rest of NSW ;,Median Price of Established House Transfers (Unstratified) ; Melbourne ;,Median Price of Established House Transfers (Unstratified) ; Rest of Vic. ;,Median Price of Established House Transfers (Unstratified) ; Brisbane ;,Median Price of Established House Transfers (Unstratified) ; Rest of Qld. ;,Median Price of Established House Transfers (Unstratified) ; Adelaide ;,Median Price of Established House Transfers (Unstratified) ; Rest of SA ;,Median Price of Established House Transfers (Unstratified) ; Perth ;,...,Number of Attached Dwelling Transfers ; Rest of Qld. ;,Number of Attached Dwelling Transfers ; Adelaide ;,Number of Attached Dwelling Transfers ; Rest of SA ;,Number of Attached Dwelling Transfers ; Perth ;,Number of Attached Dwelling Transfers ; Rest of WA ;,Number of Attached Dwelling Transfers ; Hobart ;,Number of Attached Dwelling Transfers ; Rest of Tas. ;,Number of Attached Dwelling Transfers ; Darwin ;,Number of Attached Dwelling Transfers ; Rest of NT ;,Number of Attached Dwelling Transfers ; Canberra ;
0,Unit,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,$'000,...,Number,Number,Number,Number,Number,Number,Number,Number,Number,Number
1,Series Type,Original,Original,Original,Original,Original,Original,Original,Original,Original,...,Original,Original,Original,Original,Original,Original,Original,Original,Original,Original
2,Data Type,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,...,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW,FLOW
3,Frequency,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,...,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter
4,Collection Month,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,2024-09-01 00:00:00,1410,745,825,559,890,649,818.3,465,780,...,6199,2238,296,3810,510,296,285,237,27,1014
100,2024-12-01 00:00:00,1452.5,760,852,560,935,675,850,500,820,...,5820,2191,275,3647,507,319,314,245,29,1098
101,2025-03-01 00:00:00,1450,760,840,575,945,710,855,485,820.3,...,5677,2013,239,3519,454,355,284,245,31,1088
102,2025-06-01 00:00:00,1510,770,842,575,970,725,868,510,850,...,5591,2314,226,3113,421,313,270,343,33,1185


In [14]:
print('==== INDEX ====')
print(house_data.index)
print('==== COLUMN NAMES ====')
for col in house_data.columns:
    print('COLUMN NAME:', col, 'UNITS:', house_data.loc[0, col])

==== INDEX ====
RangeIndex(start=0, stop=104, step=1)
==== COLUMN NAMES ====
COLUMN NAME: Unnamed: 0 UNITS: Unit
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Sydney ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Rest of NSW ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Melbourne ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Rest of Vic. ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Brisbane ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Rest of Qld. ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Adelaide ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Rest of SA ; UNITS: $'000
COLUMN NAME: Median Price of Established House Transfers (Unstratified) ;  Perth ; 

There are many columns in our dataset but we are interested primarily in the median price columns.

In [15]:
cleaned_house_data = house_data.copy()
cleaned_house_data.index = cleaned_house_data.pop('Unnamed: 0')
cleaned_house_data.rename_axis('Quarter', axis='rows', inplace=True)
cleaned_house_data = cleaned_house_data.iloc[9:, :]
cleaned_house_data.index = pd.to_datetime(cleaned_house_data.index)
print('==== INDEX ====')
print(cleaned_house_data.index.dtype)
cleaned_house_data

==== INDEX ====
datetime64[ns]


,Median Price of Established House Transfers (Unstratified) ; Sydney ;,Median Price of Established House Transfers (Unstratified) ; Rest of NSW ;,Median Price of Established House Transfers (Unstratified) ; Melbourne ;,Median Price of Established House Transfers (Unstratified) ; Rest of Vic. ;,Median Price of Established House Transfers (Unstratified) ; Brisbane ;,Median Price of Established House Transfers (Unstratified) ; Rest of Qld. ;,Median Price of Established House Transfers (Unstratified) ; Adelaide ;,Median Price of Established House Transfers (Unstratified) ; Rest of SA ;,Median Price of Established House Transfers (Unstratified) ; Perth ;,Median Price of Established House Transfers (Unstratified) ; Rest of WA ;,...,Number of Attached Dwelling Transfers ; Rest of Qld. ;,Number of Attached Dwelling Transfers ; Adelaide ;,Number of Attached Dwelling Transfers ; Rest of SA ;,Number of Attached Dwelling Transfers ; Perth ;,Number of Attached Dwelling Transfers ; Rest of WA ;,Number of Attached Dwelling Transfers ; Hobart ;,Number of Attached Dwelling Transfers ; Rest of Tas. ;,Number of Attached Dwelling Transfers ; Darwin ;,Number of Attached Dwelling Transfers ; Rest of NT ;,Number of Attached Dwelling Transfers ; Canberra ;
Quarter,,,,,,,,,,,,,,,,,,,,,
2002-03-01,365,NaN,241,NaN,185,NaN,166,NaN,190,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-06-01,393.5,NaN,260,NaN,182.6,NaN,175,NaN,190,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-09-01,413,NaN,265,NaN,198,NaN,181,NaN,195,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-12-01,444,NaN,280,NaN,208,NaN,195,NaN,206,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003-03-01,434.5,NaN,270,NaN,225,NaN,209.4,NaN,216,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-01,1410,745,825,559,890,649,818.3,465,780,495,...,6199,2238,296,3810,510,296,285,237,27,1014
2024-12-01,1452.5,760,852,560,935,675,850,500,820,520,...,5820,2191,275,3647,507,319,314,245,29,1098
2025-03-01,1450,760,840,575,945,710,855,485,820.3,540,...,5677,2013,239,3519,454,355,284,245,31,1088


<h2>Summary</h2>
<ul style="list-style-type: dot; font-size: 18px">
<li>Made the index of the DataFrame as the yearly quarters</li>
<li>Removed all non-price columns</li>
<li>Removed any unnecessary header rows</li>
<li>Changed the index of the data to a DateTimeIndex</li>
</ul>

In [16]:
cleaned_house_data.to_csv('../data/median_price_data.csv')

<h2>Cleaning the population data</h2>
<span style="font-size: 18px">We need population data as well, in order to calculate <strong>Net Migration per 1000 residents</strong> as a metric for the rates of migration across states.</span>

In [17]:
population_data = pd.read_excel('../data/population_states_national.xlsx', sheet_name=1)
population_data

,Unnamed: 0,Estimated Resident Population ; Male ; New South Wales ;,Estimated Resident Population ; Male ; Victoria ;,Estimated Resident Population ; Male ; Queensland ;,Estimated Resident Population ; Male ; South Australia ;,Estimated Resident Population ; Male ; Western Australia ;,Estimated Resident Population ; Male ; Tasmania ;,Estimated Resident Population ; Male ; Northern Territory ;,Estimated Resident Population ; Male ; Australian Capital Territory ;,Estimated Resident Population ; Male ; Australia ;,...,Estimated Resident Population ; Female ; Australia ;,Estimated Resident Population ; Persons ; New South Wales ;,Estimated Resident Population ; Persons ; Victoria ;,Estimated Resident Population ; Persons ; Queensland ;,Estimated Resident Population ; Persons ; South Australia ;,Estimated Resident Population ; Persons ; Western Australia ;,Estimated Resident Population ; Persons ; Tasmania ;,Estimated Resident Population ; Persons ; Northern Territory ;,Estimated Resident Population ; Persons ; Australian Capital Territory ;,Estimated Resident Population ; Persons ; Australia ;
0,Unit,Persons,Persons,Persons,Persons,Persons,Persons,Persons,Persons,Persons,...,Persons,Persons,Persons,Persons,Persons,Persons,Persons,Persons,Persons,Persons
1,Series Type,Original,Original,Original,Original,Original,Original,Original,Original,Original,...,Original,Original,Original,Original,Original,Original,Original,Original,Original,Original
2,Data Type,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,...,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE,STOCK_CLOSE
3,Frequency,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,...,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter,Quarter
4,Collection Month,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,2024-06-01 00:00:00,4225966,3432367,2759653,930989,1497454,285169,133947,235484,13503664,...,13690622,8492050,6950961,5571890,1882164,2978147,574765,260884,478430,27194286
182,2024-09-01 00:00:00,4239356,3446939,2770999,933560,1506746,285152,134544,235823,13555751,...,13744586,8518908,6982169,5595129,1887021,2995698,575024,261680,479723,27300337
183,2024-12-01 00:00:00,4247553,3457850,2782677,935534,1513924,285340,135037,236206,13596758,...,13791248,8537441,7006385,5619157,1891227,3009838,575379,262461,481121,27388006
184,2025-03-01 00:00:00,4266283,3479602,2797070,939271,1524881,285546,135769,237102,13668164,...,13864058,8575341,7051488,5647784,1898677,3030800,575959,263718,483450,27532222


In [29]:
cleaned_population_data = population_data.copy()
cleaned_population_data.index = cleaned_population_data.pop('Unnamed: 0')
persons_columns = ['Persons' in col for col in cleaned_population_data.columns]
cleaned_population_data = cleaned_population_data.loc[:, persons_columns]
new_col_names = [re.search(r';\s*([^;]+)\s*;$', col).group(1).strip() for col in cleaned_population_data.columns]
cleaned_population_data.columns = new_col_names
cleaned_population_data = cleaned_population_data.iloc[9:, :]
cleaned_population_data.rename_axis('Quarter', axis='rows', inplace=True)
cleaned_population_data.head()

,New South Wales,Victoria,Queensland,South Australia,Western Australia,Tasmania,Northern Territory,Australian Capital Territory,Australia
Quarter,,,,,,,,,
1981-06-01 00:00:00,5234889,3946917,2345208,1318769,1300056,427224,122616,227581,14923260
1981-09-01 00:00:00,5249455,3957333,2367477,1321235,1311284,427925,125186,228782,14988677
1981-12-01 00:00:00,5266894,3968398,2387943,1325176,1320221,428283,127718,229484,15054117
1982-03-01 00:00:00,5286119,3980826,2406355,1328670,1329700,429445,129593,230990,15121698
1982-06-01 00:00:00,5303580,3992870,2424586,1331108,1338899,429845,130314,233045,15184247


In [31]:
cleaned_population_data.to_csv('../data/population_data.csv')